# Generate Dimension Data

This notebook generates the master data (Dimensions): Customers, Employees, and Products.

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
import random
import os
from datetime import datetime

# Try to run common_utils.ipynb as a python script equivalent here or import if possible.
# Since we are in jupyter, we might want to use nbimporter or run `%run 00_common_utils.ipynb`
# For simplicity, we assume we run it using the %run magic.

In [ ]:
%run 00_common_utils.ipynb

## Setup Generators

In [ ]:
fake = Faker('vi_VN')
# To ensure reproducibility
Faker.seed(42)
np.random.seed(42)
random.seed(42)

NUM_CUSTOMERS = 1000
NUM_EMPLOYEES = 50
NUM_PRODUCTS = 500

## Generate DIM_PRODUCTS

In [ ]:
def generate_products(n):
    data = []
    template_path = PROJECT_ROOT / 'templates' / 'MauFileSanPham.xlsx'
    
    # Danh sách các mặt hàng phổ biến ở tạp hóa Hà Nội
    hanoi_common_products = [
        ("Bia Hà Nội lon 330ml", "Đồ uống>>Bia"),
        ("Bia 333 lon 330ml", "Đồ uống>>Bia"),
        ("Mì Hảo Hảo tôm chua cay", "Thực phẩm>>Mì ăn liền"),
        ("Phở bò Cung Đình", "Thực phẩm>>Mì ăn liền"),
        ("Nước mắm Nam Ngư Đệ Nhị 900ml", "Thực phẩm>>Gia vị"),
        ("Tương ớt Chinsu 250g", "Thực phẩm>>Gia vị"),
        ("Dầu ăn Neptune 1L", "Thực phẩm>>Gia vị"),
        ("Bột canh Hải Châu", "Thực phẩm>>Gia vị"),
        ("Sữa tươi Vinamilk có đường 180ml", "Khác>>Sữa"),
        ("Sữa chua Mộc Châu", "Khác>>Sữa chua"),
        ("Trứng gà Ba Huân", "Thực phẩm>>Tươi sống"),
        ("Xúc xích CP tiệt trùng", "Thực phẩm>>Đồ hộp"),
        ("Bánh Choco-Pie hộp 12 cái", "Kẹo bánh"),
        ("Kẹo dừa Bến Tre", "Kẹo bánh"),
        ("Kẹo lạc vừng", "Kẹo bánh"),
        ("Gạo Tám Hải Hậu 10kg", "Thực phẩm>>Gạo"),
        ("Giấy vệ sinh Hà Nội 3 lớp", "Hàng hóa>>Vệ sinh"),
        ("Kem đánh răng P/S trà xanh", "Mỹ phẩm>>Chăm sóc răng miệng"),
        ("Dầu gội Clear bạc hà", "Mỹ phẩm>>Chăm sóc tóc"),
        ("Băng vệ sinh Diana", "Hàng hóa>>Vệ sinh cá nhân"),
        ("Nước giặt OMO Matic", "Hàng hóa>>Hóa phẩm"),
        ("Nước rửa chén Sunlight chanh", "Hàng hóa>>Hóa phẩm"),
        ("Thẻ cào Viettel 50k", "Dịch vụ>>Thẻ cào"),
        ("Thẻ cào Mobifone 50k", "Dịch vụ>>Thẻ cào"),
        ("Nước khoáng Lavie 500ml", "Đồ uống>>Nước lọc"),
        ("Trà xanh Không Độ", "Đồ uống>>Trà đóng chai"),
        ("Coca Cola lon 330ml", "Đồ uống>>Nước ngọt"),
        ("Bò húc Redbull", "Đồ uống>>Nước tăng lực")
    ]
    
    sample_items = []
    try:
        df_tpl = pd.read_excel(template_path)
        for _, row in df_tpl.iterrows():
            if pd.notna(row['Tên hàng']):
                cat = row['Nhóm hàng(3 Cấp)'] if pd.notna(row['Nhóm hàng(3 Cấp)']) else 'Mặc định'
                sample_items.append((row['Tên hàng'], cat))
    except Exception as e:
        logger.warning(f'Không thể đọc template: {e}')
        
    # Kết hợp dữ liệu từ template và dữ liệu phổ biến Hà Nội
    all_sample_items = list(set(sample_items + hanoi_common_products))
    
    for i in range(1, n + 1):
        item = random.choice(all_sample_items)
        base_name = item[0]
        category = item[1]
        
        # Thêm hậu tố ngẫu nhiên để tránh trùng lặp nếu số lượng lớn
        product_name = f"{base_name} - Mẫu {i:03d}" if n > len(all_sample_items) else base_name
        cost = random.randint(10, 500) * 1000
        sale = int(cost * random.uniform(1.2, 2.0))
            
        data.append({
            'product_code': f'SP{i:05d}',
            'product_name': product_name,
            'category': category,
            'cost_price': cost,
            'sale_price': sale,
            'stock_on_hand': random.randint(50, 500)
        })
    return pd.DataFrame(data)

dim_products = generate_products(NUM_PRODUCTS)
logger.info(f"Generated {len(dim_products)} products.")
dim_products.head()

## Generate DIM_CUSTOMERS

In [ ]:
def generate_customers(n):
    data = []
    for i in range(1, n + 1):
        data.append({
            'customer_code': f'KH{i:05d}',
            'customer_name': fake.name(),
            'phone': fake.phone_number(),
            'address': fake.address()
        })
    return pd.DataFrame(data)

dim_customers = generate_customers(NUM_CUSTOMERS)
logger.info(f"Generated {len(dim_customers)} customers.")
dim_customers.head()

## Generate DIM_EMPLOYEES

In [ ]:
def generate_employees(n):
    data = []
    roles = ['Thu ngân', 'Quản lý', 'Kho', 'Bán hàng']
    branches = ['Chi nhánh Trung tâm', 'Chi nhánh Q1', 'Chi nhánh Q2', 'Chi nhánh Q3']
    for i in range(1, n + 1):
        data.append({
            'employee_code': f'NV{i:04d}',
            'employee_name': fake.name(),
            'role': random.choice(roles),
            'branch': random.choice(branches)
        })
    return pd.DataFrame(data)

dim_employees = generate_employees(NUM_EMPLOYEES)
logger.info(f"Generated {len(dim_employees)} employees.")
dim_employees.head()

## Export Data

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save to warehouse as CSV
dim_products.to_csv(WAREHOUSE_DIR / f'dim_products_{timestamp}.csv', index=False)
dim_customers.to_csv(WAREHOUSE_DIR / f'dim_customers_{timestamp}.csv', index=False)
dim_employees.to_csv(WAREHOUSE_DIR / f'dim_employees_{timestamp}.csv', index=False)

# Save as Excel for KiotViet import
export_chunks_to_excel(dim_products, 'Products_Import', EXCEL_OUT_DIR, chunk_size=1000, column_mapping=PRODUCT_COL_MAP)
export_chunks_to_excel(dim_customers, 'Customers_Import', EXCEL_OUT_DIR, chunk_size=1000, column_mapping=CUSTOMER_COL_MAP)
export_chunks_to_excel(dim_employees, 'Employees_Import', EXCEL_OUT_DIR, chunk_size=1000, column_mapping=EMPLOYEE_COL_MAP)

logger.info("Dimension data generation and export complete!")